# KrishiNexus — `krishi-bangla-lora-v1` QLoRA training

Fine-tunes **Qwen2.5-1.5B-Instruct** on the generated Bangla agri-advisor SFT set with QLoRA, then merges the adapter.
Recipe follows `TRAINING_PLAN.md` section 4 (TRL 1.x). Runs on a free Colab T4, or a local 4 GB GPU per the plan's Windows notes.
For the full run (frozen-test evaluation, base vs adapter, GGUF export) use `kaggle/krishinexus_lora_kaggle.ipynb`.

**Frozen-test rule:** training reads only `train.jsonl` + `val.jsonl`. `test.jsonl` is opened once, later, by the eval script — never here.


## 1. GPU check


In [ ]:
import torch
assert torch.cuda.is_available(), 'CPU only: switch the runtime to a GPU'
# bf16 needs Ampere or newer (RTX 30xx, L4, A100); the T4 on Colab's free tier is sm_75 and trains in fp16.
BF16 = torch.cuda.get_device_capability(0) >= (8, 0)
DTYPE = torch.bfloat16 if BF16 else torch.float16
print('Device:', torch.cuda.get_device_name(0), '| precision:', 'bf16' if BF16 else 'fp16')


## 2. Install

On Colab, torch is preinstalled — this adds the training stack. On local Windows (RTX 3050 4 GB) first install the
cu128 torch wheel per `TRAINING_PLAN.md` section 4, then run this with `--no-deps` for torch.


In [ ]:
%pip install -q -U "transformers>=5" "trl>=1.16,<2" "peft>=0.17" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.50.2"


## 3. Get the data

The SFT set is **generated from the repo** (it is synthetic — it distils the deterministic policy), not downloaded.
Cloning and running the exporter reproduces `train/val/test.jsonl` deterministically (seed 20261022).

If you already have the three JSONL files, put them in `KrishiNexus/training/data/` and skip the clone.


In [ ]:
import os, subprocess, sys, pathlib
REPO = 'https://github.com/HTanvir0011/KrishiNexus.git'
if not pathlib.Path('KrishiNexus').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
# prepare_data.py imports the backend generators and writes training/data/{train,val,test}.jsonl
os.chdir('KrishiNexus')  # the published repo has backend/ and training/ at its root
%pip install -q "pydantic>=2" httpx numpy python-dotenv  # what prepare_data.py imports from the backend
subprocess.run([sys.executable, 'training/prepare_data.py'], check=True)
DATA = pathlib.Path('training/data')
print(sorted(p.name for p in DATA.glob('*.jsonl')))


## 4. Load train + val as prompt/completion rows

Each exported row is `{messages:[system,user,assistant]}`. We split it into `prompt=[system,user]` and
`completion=[assistant]` so TRL applies **completion-only loss** by default (section 4).


In [ ]:
from datasets import load_dataset

def to_prompt_completion(ex):
    msgs = ex['messages']
    return {'prompt': msgs[:-1], 'completion': msgs[-1:]}

train_ds = load_dataset('json', data_files='training/data/train.jsonl', split='train').map(to_prompt_completion, remove_columns=['messages'])
val_ds   = load_dataset('json', data_files='training/data/val.jsonl',   split='train').map(to_prompt_completion, remove_columns=['messages'])
print(len(train_ds), 'train /', len(val_ds), 'val')
print(train_ds[0]['prompt'][-1]['content'][:120])


## 5. Model, tokenizer, 4-bit NF4 config


In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
)


## 6. Train (QLoRA, rank 16)

Hyper-parameters and the 7 target modules are exactly `lora_adapter_spec`. ~430 optimizer steps; ~1.5–3 h on the 4 GB
card, faster on a T4. `test.jsonl` is not touched.


In [ ]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, dtype=DTYPE,
                                             attn_implementation='sdpa', device_map={'': 0})  # sdpa: no flash-attn/triton on Windows

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

cfg = SFTConfig(
    output_dir='krishi-bangla-lora-v1',
    max_length=512,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=13,  # ~3% of the ~432 optimizer steps; transformers 5 removed warmup_ratio
    optim='paged_adamw_8bit',
    bf16=BF16, fp16=not BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=0,
    logging_steps=10,
    eval_strategy='steps', eval_steps=50,
    save_strategy='steps', save_steps=50, save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)
if not BF16:  # TRL casts QLoRA adapter weights to bf16; without bf16 hardware keep them fp32 for fp16 mixed precision
    for param in trainer.model.parameters():
        if param.requires_grad:
            param.data = param.data.float()
trainer.train()


## 7. Save and merge the adapter

Saves the LoRA adapter, then merges into the bf16 base for GGUF export (section 5).


In [ ]:
trainer.save_model('krishi-bangla-lora-v1')  # adapter only

from peft import AutoPeftModelForCausalLM
merged = AutoPeftModelForCausalLM.from_pretrained('krishi-bangla-lora-v1', dtype=torch.bfloat16)
merged = merged.merge_and_unload()
merged.save_pretrained('krishi-bangla-merged')
tokenizer.save_pretrained('krishi-bangla-merged')
print('merged ->', os.path.abspath('krishi-bangla-merged'))


## 8. GGUF export + Ollama (run in a shell, section 5)

Ollama ≥ 0.34.2 dropped adapter loading, so deploy the **merged** model as GGUF:

```bash
git clone https://github.com/ggml-org/llama.cpp
pip install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
python llama.cpp/convert_hf_to_gguf.py krishi-bangla-merged --outtype f16 --outfile krishi-bangla-f16.gguf
# llama-quantize from the llama.cpp release assets (llama-b<NNNN>-bin-win-cpu-x64.zip):
llama-quantize krishi-bangla-f16.gguf krishi-bangla-q4_K_M.gguf Q4_K_M
```

Modelfile:
```
FROM ./krishi-bangla-q4_K_M.gguf
PARAMETER temperature 0.2
SYSTEM You are KrishiNexus, a Bangla-speaking agricultural market and storage advisor.
```
`ollama create krishi-bangla -f Modelfile`


## 9. Evaluation — separate step, frozen test split

Do **not** evaluate here. `test.jsonl` (660 examples, incl. the Chittagonian / holding_months=7 / Khaliajuri OOD
holdouts) is opened once by the deterministic eval script per `TRAINING_PLAN.md` section 6, which replays each
example's inputs through the same engines and compares base vs adapter. Build `training/eval_lora.py` for that run.
